# 실습 9: Ollama LLM을 활용한 HTTP 분류 (2교시)

특성 추출 없이 **HTTP 요청 텍스트를 그대로** Ollama gemma3:4b에 보여주고
정상/공격을 분류합니다.

**사전 조건**:
- Ollama 서버 실행 중 (`ollama serve`)
- `gemma3:4b` 모델 다운로드됨 (`ollama list`로 확인)
- 7주차 `processed_data.pkl` 존재 (LLM용 텍스트 샘플 포함)


In [12]:
# %% [Setup] 패키지 import 및 LLM용 샘플 로드
import pickle
import time
import json
import re
import pandas as pd
from urllib.parse import unquote
from sklearn.metrics import accuracy_score, f1_score, classification_report

import ollama  # pip install ollama

with open("processed_data.pkl", "rb") as f:
    data = pickle.load(f)

llm_sample = data["llm_sample"].head(100).reset_index(drop=True)
print(f"분류 대상: {len(llm_sample)}건")
print(f"라벨 분포: 정상 {(llm_sample.get('is_attack',0)==0).sum()}건 / "
      f"공격 {(llm_sample.get('is_attack',0)==1).sum()}건")


분류 대상: 100건
라벨 분포: 정상 56건 / 공격 44건


## 1. 분류 프롬프트 설계

LLM 응답을 안정적으로 파싱하기 위해:
- **Few-shot 예시** 2개로 출력 형식을 학습시킴
- **JSON 형태**로 응답하도록 강제
- 영어 프롬프트(gemma3:4b가 영어에 더 정확)

In [14]:
# %% [1] HTTP 텍스트 재구성 + 프롬프트 함수
def build_http_text(row) -> str:
    method = row.get("method", "GET")
    url    = unquote(str(row.get("url", "")), encoding="latin-1")
    body   = str(row.get("body_decoded", row.get("body", "")) or "")
    text   = f"{method} {url} HTTP/1.1"
    if body and body != "nan":
        text += f"\nBody: {body[:200]}"
    return text


PROMPT_TEMPLATE = 'You are a web security expert. Classify each HTTP request as "Normal" or "Anomalous" and provide a brief reason.\n\nExamples:\nRequest: GET /index.jsp HTTP/1.1\nOutput: {{"label": "Normal", "reason": "Standard page request, no suspicious pattern"}}\n\nRequest: GET /search?q=\' OR \'1\'=\'1 HTTP/1.1\nOutput: {{"label": "Anomalous", "reason": "Classic SQL Injection pattern with OR 1=1"}}\n\nNow classify:\nRequest: {http_text}\nOutput:'


def classify_with_llm(http_text: str, model: str = "gemma3:1b") -> dict:
    """Ollama로 HTTP 요청 분류 -> {label, reason}"""
    prompt = PROMPT_TEMPLATE.format(http_text=http_text)
    response = ollama.chat(
        model=model,
        messages=[{"role":"user","content":prompt}],
        options={"temperature": 0},  # 결정성 높이기
    )
    text = response["message"]["content"]

    # JSON 추출 - LLM이 가끔 앞뒤 설명을 붙임
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        return {"label":"Unknown", "reason": text[:80]}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"label":"Unknown", "reason": text[:80]}


# 단건 테스트
test_text = "GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1"
print("입력:", test_text)
print("응답:", classify_with_llm(test_text))


입력: GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1
응답: {'label': 'Anomalous', 'reason': 'Attempting to inject a shell command via a URL parameter. The `+` character is often used for command injection.'}


## 2. 100건 분류 + 시간 측정

CPU 환경 기준 건당 1~3초가 표준. 100건 ≈ 2~5분 소요.

In [15]:
# %% [2] 100건 분류
results = []
start = time.time()

for i, row in llm_sample.iterrows():
    http_text = build_http_text(row)
    result = classify_with_llm(http_text)
    true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
    results.append({
        "idx": i,
        "true": true_label,
        "pred": result.get("label", "Unknown"),
        "reason": result.get("reason", "")[:120],
        "http_short": http_text[:100],
    })
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(llm_sample)}건 완료 "
              f"({elapsed:.1f}초, 건당 {elapsed/(i+1):.2f}초)")

llm_time = time.time() - start
llm_df = pd.DataFrame(results)
print(f"\n총 소요: {llm_time:.1f}초")
print(f"1만 건 환산: 약 {llm_time/100*10000/60:.0f}분")


  10/100건 완료 (10.3초, 건당 1.03초)
  20/100건 완료 (18.8초, 건당 0.94초)
  30/100건 완료 (27.3초, 건당 0.91초)
  40/100건 완료 (36.9초, 건당 0.92초)
  50/100건 완료 (47.4초, 건당 0.95초)
  60/100건 완료 (59.3초, 건당 0.99초)
  70/100건 완료 (69.6초, 건당 0.99초)
  80/100건 완료 (80.5초, 건당 1.01초)
  90/100건 완료 (92.8초, 건당 1.03초)
  100/100건 완료 (102.6초, 건당 1.03초)

총 소요: 102.6초
1만 건 환산: 약 171분


In [16]:
# %% [3] 정확도/F1 계산
llm_df["pred_clean"] = llm_df["pred"].replace({"Unknown":"Normal"})
y_true = (llm_df["true"] == "Anomalous").astype(int)
y_pred = (llm_df["pred_clean"] == "Anomalous").astype(int)

llm_acc = accuracy_score(y_true, y_pred)
llm_f1  = f1_score(y_true, y_pred)

print(f"LLM 정확도: {llm_acc:.4f}")
print(f"LLM F1:    {llm_f1:.4f}")
print(f"분류 실패(Unknown): {(llm_df['pred']=='Unknown').sum()}건")
print()
print(classification_report(y_true, y_pred, target_names=["Normal","Anomalous"]))


LLM 정확도: 0.6900
LLM F1:    0.6173
분류 실패(Unknown): 0건

              precision    recall  f1-score   support

      Normal       0.70      0.79      0.74        56
   Anomalous       0.68      0.57      0.62        44

    accuracy                           0.69       100
   macro avg       0.69      0.68      0.68       100
weighted avg       0.69      0.69      0.69       100



## 3. 자연어 판단 근거 검토 ★

LLM의 가장 큰 강점: **왜 그렇게 판단했는지** 사람이 읽을 수 있는 문장으로 설명.
이는 SOC(보안관제) 분석가가 1차 분류를 검토할 때 매우 유용합니다.

In [17]:
# %% [4] 공격으로 판단한 사례 + LLM 근거
print("=== LLM이 공격으로 판단한 사례 (상위 5건) ===\n")
attack_pred = llm_df[llm_df["pred"] == "Anomalous"].head(5)
for _, r in attack_pred.iterrows():
    correct = "OK" if r["true"] == "Anomalous" else "오탐"
    print(f"[{correct}] 실제={r['true']:10s}  요청: {r['http_short']}")
    print(f"   - LLM 근거: {r['reason']}\n")


=== LLM이 공격으로 판단한 사례 (상위 5건) ===

[OK] 실제=Anomalous   요청: GET /tienda1/miembros/editar.jsp?modo=registro&loginA=lieure&password=rEbatible&nombre=Tarciano&apel
   - LLM 근거: The request includes a complex set of parameters, including a `B` parameter (likely a B-flag for a B-tree, which is ofte

[오탐] 실제=Normal      요청: POST /tienda1/publico/entrar.jsp HTTP/1.1
Body: errorMsg=Credenciales+incorrectas
   - LLM 근거: Sending sensitive data (credentials) in the POST request, which is a common indicator of a potential attack.

[OK] 실제=Anomalous   요청: POST /tienda1/publico/vaciar.jsp HTTP/1.1
Body: B2=Vaciar+carrito%3CSCRIPT%3Ealert%28%22Paros%22%29%
   - LLM 근거: Sending a POST request with a malicious payload (B2=Vaciar+carrito%3B... and a JavaScript alert) is a strong indicator o

[OK] 실제=Anomalous   요청: POST /tienda1/publico/vaciar.jsp HTTP/1.1
Body: B2A=Vaciar+carrito
   - LLM 근거: The request body contains a seemingly innocuous value ('Vaciar+carrito') that is likely intended to be used for i

In [18]:
# %% [5] 결과 저장 (3교시에서도 활용)
with open("llm_classification_results.pkl", "wb") as f:
    pickle.dump({
        "llm_df": llm_df,
        "llm_acc": llm_acc,
        "llm_f1": llm_f1,
        "llm_time": llm_time,
        "n_samples": len(llm_sample),
    }, f)
print(">> llm_classification_results.pkl 저장 완료")


>> llm_classification_results.pkl 저장 완료


**다음**: `comparison_analysis.ipynb`로 1교시 ML 결과와 종합 비교합니다.

## 4. 프롬프트 변형 비교 실험

동일한 50건 샘플로 4가지 프롬프트 전략을 비교합니다.

| 버전 | 전략 |
|------|------|
| v1_baseline | few-shot 2개 (기존) |
| v2_more_shot | few-shot 6개 (XSS, Path Traversal 추가) |
| v3_cot | Chain-of-Thought (단계별 분석 후 판단) |
| v4_pattern | 공격 패턴 명시 (어떤 패턴이 공격인지 설명 포함) |

In [19]:
# %% [6] 프롬프트 버전 정의
PROMPT_V1 = (
    'You are a web security expert. Classify each HTTP request as "Normal" or "Anomalous" '
    'and provide a brief reason.\n\n'
    'Examples:\n'
    'Request: GET /index.jsp HTTP/1.1\n'
    'Output: {{"label": "Normal", "reason": "Standard page request, no suspicious pattern"}}\n\n'
    'Request: GET /search?q=\' OR \'1\'=\'1 HTTP/1.1\n'
    'Output: {{"label": "Anomalous", "reason": "Classic SQL Injection pattern with OR 1=1"}}\n\n'
    'Now classify:\n'
    'Request: {http_text}\n'
    'Output:'
)

PROMPT_V2 = (
    'You are a web security expert. Classify each HTTP request as "Normal" or "Anomalous" '
    'and provide a brief reason.\n\n'
    'Examples:\n'
    'Request: GET /index.jsp HTTP/1.1\n'
    'Output: {{"label": "Normal", "reason": "Standard page request, no suspicious pattern"}}\n\n'
    'Request: GET /search?q=\' OR \'1\'=\'1 HTTP/1.1\n'
    'Output: {{"label": "Anomalous", "reason": "SQL Injection pattern with OR 1=1"}}\n\n'
    'Request: GET /products?category=electronics HTTP/1.1\n'
    'Output: {{"label": "Normal", "reason": "Normal category filter parameter"}}\n\n'
    'Request: GET /page?file=../../etc/passwd HTTP/1.1\n'
    'Output: {{"label": "Anomalous", "reason": "Path traversal attempting to access system files"}}\n\n'
    'Request: POST /login HTTP/1.1\nBody: user=admin&pass=secret\n'
    'Output: {{"label": "Normal", "reason": "Standard login form submission"}}\n\n'
    'Request: GET /search?q=<script>alert(1)</script> HTTP/1.1\n'
    'Output: {{"label": "Anomalous", "reason": "XSS attack with script tag injection"}}\n\n'
    'Now classify:\n'
    'Request: {http_text}\n'
    'Output:'
)

PROMPT_V3 = (
    'You are a web security expert. Analyze the HTTP request step by step, then classify it.\n\n'
    'Step 1: Identify the HTTP method and endpoint.\n'
    'Step 2: Check URL parameters for suspicious patterns (SQL injection, XSS, path traversal, command injection).\n'
    'Step 3: Check request body if present.\n'
    'Step 4: Make a final classification.\n\n'
    'Request: {http_text}\n\n'
    'After analysis, output ONLY this JSON (no other text):\n'
    '{{"label": "Normal" or "Anomalous", "reason": "one sentence explanation"}}'
)

PROMPT_V4 = (
    'You are a web security expert. Classify the HTTP request below.\n\n'
    'Attack patterns to detect:\n'
    "- SQL Injection: OR 1=1, UNION SELECT, --, ;DROP, quotes in params\n"
    "- XSS: <script>, javascript:, alert(), onerror=\n"
    "- Path Traversal: ../, ..\\, /etc/passwd, /windows/system32\n"
    "- Command Injection: ;ls, |cat, &&whoami, `id`\n"
    "- Encoding tricks: %27 (quote), %3C%3E (angle brackets), %00 (null byte)\n\n"
    'If none of these patterns exist, classify as Normal.\n\n'
    'Request: {http_text}\n\n'
    'Output ONLY valid JSON:\n'
    '{{"label": "Normal" or "Anomalous", "reason": "one sentence explanation"}}'
)

PROMPT_VERSIONS = {
    "v1_baseline":  PROMPT_V1,
    "v2_more_shot": PROMPT_V2,
    "v3_cot":       PROMPT_V3,
    "v4_pattern":   PROMPT_V4,
}
print("프롬프트 버전 정의 완료:", list(PROMPT_VERSIONS.keys()))

프롬프트 버전 정의 완료: ['v1_baseline', 'v2_more_shot', 'v3_cot', 'v4_pattern']


In [20]:
# %% [7] 4가지 프롬프트 비교 실행 (50건, 모델: gemma3:1b)
# 주의: 4 x 50 = 200회 LLM 호출 → 약 5~10분 소요

COMPARE_N   = 50
MODEL_NAME  = "gemma3:1b"
compare_sample = llm_sample.head(COMPARE_N).reset_index(drop=True)

def classify_with_prompt(http_text: str, prompt_template: str, model: str) -> dict:
    prompt = prompt_template.format(http_text=http_text)
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    text = response["message"]["content"]
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        return {"label": "Unknown", "reason": text[:80]}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"label": "Unknown", "reason": text[:80]}


comparison_results = {}

for version_name, prompt_template in PROMPT_VERSIONS.items():
    print(f"\n[{version_name}] 분류 시작...")
    rows, start = [], time.time()

    for i, row in compare_sample.iterrows():
        http_text = build_http_text(row)
        result = classify_with_prompt(http_text, prompt_template, MODEL_NAME)
        true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
        rows.append({
            "true": true_label,
            "pred": result.get("label", "Unknown"),
            "reason": result.get("reason", "")[:120],
        })
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{COMPARE_N}건 완료 ({time.time()-start:.1f}초)")

    elapsed = time.time() - start
    df = pd.DataFrame(rows)
    df["pred_clean"] = df["pred"].replace({"Unknown": "Normal"})
    y_true = (df["true"] == "Anomalous").astype(int)
    y_pred = (df["pred_clean"] == "Anomalous").astype(int)

    comparison_results[version_name] = {
        "df":      df,
        "acc":     accuracy_score(y_true, y_pred),
        "f1":      f1_score(y_true, y_pred, zero_division=0),
        "unknown": (df["pred"] == "Unknown").sum(),
        "time_s":  elapsed,
        "prompt":  prompt_template,
    }
    print(f"  -> 정확도: {comparison_results[version_name]['acc']:.4f}  "
          f"F1: {comparison_results[version_name]['f1']:.4f}  "
          f"Unknown: {comparison_results[version_name]['unknown']}건  "
          f"소요: {elapsed:.1f}초")

print("\n=== 비교 완료 ===")


[v1_baseline] 분류 시작...
  10/50건 완료 (11.5초)
  20/50건 완료 (20.4초)
  30/50건 완료 (30.4초)
  40/50건 완료 (41.0초)
  50/50건 완료 (51.7초)
  -> 정확도: 0.6800  F1: 0.6190  Unknown: 0건  소요: 51.7초

[v2_more_shot] 분류 시작...
  10/50건 완료 (9.6초)
  20/50건 완료 (18.6초)
  30/50건 완료 (28.2초)
  40/50건 완료 (38.8초)
  50/50건 완료 (51.9초)
  -> 정확도: 0.7800  F1: 0.7556  Unknown: 0건  소요: 51.9초

[v3_cot] 분류 시작...
  10/50건 완료 (21.4초)
  20/50건 완료 (39.6초)
  30/50건 완료 (58.0초)
  40/50건 완료 (77.0초)
  50/50건 완료 (96.2초)
  -> 정확도: 0.5600  F1: 0.2143  Unknown: 1건  소요: 96.2초

[v4_pattern] 분류 시작...
  10/50건 완료 (14.5초)
  20/50건 완료 (29.4초)
  30/50건 완료 (45.2초)
  40/50건 완료 (62.0초)
  50/50건 완료 (79.8초)
  -> 정확도: 0.5400  F1: 0.0800  Unknown: 0건  소요: 79.8초

=== 비교 완료 ===


In [21]:
# %% [8] 결과 비교표 출력
print("=" * 65)
print(f"{'버전':<16} {'정확도':>8} {'F1':>8} {'Unknown':>8} {'소요(초)':>10}")
print("-" * 65)
for name, res in comparison_results.items():
    print(f"{name:<16} {res['acc']:>8.4f} {res['f1']:>8.4f} "
          f"{res['unknown']:>8} {res['time_s']:>10.1f}")
print("=" * 65)

best = max(comparison_results, key=lambda k: comparison_results[k]["f1"])
print(f"\n최고 F1 버전: [{best}]  F1={comparison_results[best]['f1']:.4f}")

버전                    정확도       F1  Unknown      소요(초)
-----------------------------------------------------------------
v1_baseline        0.6800   0.6190        0       51.7
v2_more_shot       0.7800   0.7556        0       51.9
v3_cot             0.5600   0.2143        1       96.2
v4_pattern         0.5400   0.0800        0       79.8

최고 F1 버전: [v2_more_shot]  F1=0.7556


In [22]:
# %% [9] MD 보고서 자동 생성
import datetime

now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

lines = []
lines.append("# LLM HTTP 분류 프롬프트 비교 실험 보고서\n")
lines.append(f"- **실험일**: {now_str}")
lines.append(f"- **모델**: `{MODEL_NAME}` (Ollama)")
lines.append(f"- **평가 샘플**: {COMPARE_N}건 (정상/공격 혼합)\n")

lines.append("## 1. 프롬프트 전략 요약\n")
strategy_desc = {
    "v1_baseline":  "few-shot 2개 (정상 1, SQL Injection 1) — 기본 JSON 출력 지시",
    "v2_more_shot": "few-shot 6개 (정상 2, SQL Injection / Path Traversal / XSS / POST 로그인) — 다양한 공격 예시 추가",
    "v3_cot":       "Chain-of-Thought — 4단계 분석(메서드→URL→Body→판단) 후 JSON 출력",
    "v4_pattern":   "공격 패턴 명시 — SQL Injection·XSS·Path Traversal·Command Injection·인코딩 트릭 목록 제공",
}
for name, desc in strategy_desc.items():
    lines.append(f"- **{name}**: {desc}")

lines.append("\n## 2. 실험 결과\n")
lines.append(f"| 버전 | 정확도 | F1 Score | Unknown | 소요 시간(초) |")
lines.append(f"|------|--------|----------|---------|--------------|")
for name, res in comparison_results.items():
    marker = " ★" if name == best else ""
    lines.append(
        f"| {name}{marker} | {res['acc']:.4f} | {res['f1']:.4f} "
        f"| {res['unknown']} | {res['time_s']:.1f} |"
    )

lines.append(f"\n**최고 성능 버전**: `{best}` (F1 = {comparison_results[best]['f1']:.4f})\n")

lines.append("## 3. 프롬프트 전문\n")
for name, res in comparison_results.items():
    lines.append(f"### {name}\n")
    lines.append("```")
    lines.append(res["prompt"].replace("{http_text}", "<HTTP_REQUEST_HERE>"))
    lines.append("```\n")

lines.append("## 4. 분석 및 결론\n")
lines.append("- few-shot 예시가 많을수록 모델이 공격 패턴을 더 잘 인식하는 경향이 있습니다.")
lines.append("- Chain-of-Thought 방식은 추론 과정을 명시해 정확도가 높아질 수 있으나 응답 길이가 길어져 JSON 파싱 실패(Unknown) 비율이 증가할 수 있습니다.")
lines.append("- 공격 패턴 명시(v4) 방식은 모델이 놓치기 쉬운 인코딩 우회 기법까지 커버합니다.")
lines.append("- 실제 배포 시에는 Unknown 비율, 처리 속도, F1을 종합적으로 고려해야 합니다.\n")

md_text = "\n".join(lines)
md_path = "llm_prompt_comparison.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_text)

print(f"MD 파일 저장 완료: {md_path}")
print(f"총 {len(lines)}줄")
print("\n--- 미리보기 (처음 20줄) ---")
print("\n".join(lines[:20]))

MD 파일 저장 완료: llm_prompt_comparison.md
총 39줄

--- 미리보기 (처음 20줄) ---
# LLM HTTP 분류 프롬프트 비교 실험 보고서

- **실험일**: 2026-05-15 15:49
- **모델**: `gemma3:1b` (Ollama)
- **평가 샘플**: 50건 (정상/공격 혼합)

## 1. 프롬프트 전략 요약

- **v1_baseline**: few-shot 2개 (정상 1, SQL Injection 1) — 기본 JSON 출력 지시
- **v2_more_shot**: few-shot 6개 (정상 2, SQL Injection / Path Traversal / XSS / POST 로그인) — 다양한 공격 예시 추가
- **v3_cot**: Chain-of-Thought — 4단계 분석(메서드→URL→Body→판단) 후 JSON 출력
- **v4_pattern**: 공격 패턴 명시 — SQL Injection·XSS·Path Traversal·Command Injection·인코딩 트릭 목록 제공

## 2. 실험 결과

| 버전 | 정확도 | F1 Score | Unknown | 소요 시간(초) |
|------|--------|----------|---------|--------------|
| v1_baseline | 0.6800 | 0.6190 | 0 | 51.7 |
| v2_more_shot ★ | 0.7800 | 0.7556 | 0 | 51.9 |
| v3_cot | 0.5600 | 0.2143 | 1 | 96.2 |
| v4_pattern | 0.5400 | 0.0800 | 0 | 79.8 |

**최고 성능 버전**: `v2_more_shot` (F1 = 0.7556)

## 3. 프롬프트 전문

### v1_baseline

```


In [23]:
# %% [10] GitHub 업로드 (git add → commit → push)
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # 프로젝트 루트
md_abs    = os.path.abspath(md_path)
nb_abs    = os.path.abspath("llm_classification.ipynb")

def run(cmd, cwd=repo_root):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print("[stderr]", result.stderr.strip())
    return result.returncode

print("== git add ==")
run(f'git add "{md_abs}" "{nb_abs}"')

print("\n== git commit ==")
run('git commit -m "10week: LLM prompt comparison experiment (4 strategies, gemma3:1b)"')

print("\n== git push ==")
run("git push")

print("\n업로드 완료!")

== git add ==

== git commit ==
[main 78726335] 10week: LLM prompt comparison experiment (4 strategies, gemma3:1b)
 2 files changed, 408 insertions(+)
 create mode 100644 source/10_week/llm_classification.ipynb
 create mode 100644 source/10_week/llm_prompt_comparison.md

== git push ==
[stderr] To https://github.com/wamong/bigdata-project.git
   d1ea5643..78726335  main -> main

업로드 완료!
